In [1]:
%pip install -r requirements-offline.txt --quiet

/Users/utarn/projects/datascience-metrology/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Solution — Module M3: ข้อมูลไม่สะอาด

> **เฉลย — ผู้สอนจะเปิดเผยในห้องหลังจบช่วงแล็บ** ลองทำ `exercise.ipynb` ให้สุดก่อนเปิดไฟล์นี้

**ข้อมูล (citation):** เดียวกับ `exercise.ipynb`

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit("ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)")


DATA = repo_root() / "datasets" / "adult"


## ส่วนที่ 1 — ค่าว่างและค่าปลอม (Adult)

In [3]:
cols = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]
adult = pd.read_csv(DATA / "adult.data", header=None, names=cols, skipinitialspace=True)
adult.shape

(32561, 15)

In [4]:
adult = adult.replace("?", np.nan)
adult.isna().sum().loc[lambda s: s > 0]

workclass         1836
occupation        1843
native_country     583
dtype: int64

**1.1** `workclass` 1,836 (5.6%), `occupation` 1,843 (5.7%), `native_country` 583 (1.8%)

**1.2** ดูคำตอบในเซลล์ถัดไป — เสียแค่ ~5.6% จึง "ตัดได้" ตามกฎ ~10%

In [5]:
adult_drop = adult.dropna(subset=["workclass", "occupation"])
print(f"เหลือ {len(adult_drop):,} จาก {len(adult):,} แถว "
      f"(เสีย {(1 - len(adult_drop) / len(adult)):.1%})")
# 5.6% — ต่ำกว่าเกณฑ์ ~10% จึงตัดทิ้งได้อย่างมีเหตุผล
# ถ้าคำถามวิเคราะห์ผูกกับ native_country ให้พิจารณาแยก เพราะค่าหายของคอลัมน์นั้นไม่ใช่แถวเดียวกัน

เหลือ 30,718 จาก 32,561 แถว (เสีย 5.7%)


## ส่วนที่ 2 — สะกดไม่ตรงและหน่วยปน (Coffee)

In [6]:
coffee = pd.read_csv(repo_root() / "datasets" / "coffee_quality" / "arabica_data_cleaned.csv")
coffee["altitude_m"] = (
    coffee["Altitude"].astype("string").str.extract(r"(\d+\.?\d*)").astype(float)
)
print("แปลงได้:", coffee["altitude_m"].notna().sum(), "จาก", coffee["Altitude"].notna().sum())

แปลงได้: 1085 จาก 1088


In [7]:
coffee["Variety"].value_counts().loc[lambda s: s < 15]

Variety
Pacas                    13
Gesha                    12
SL34                      8
Pacamara                  8
Peaberry                  5
Arusha                    5
Sumatra                   3
Mandheling                3
Ethiopian Yirgacheffe     2
Java                      2
Ruiru 11                  2
Blue Mountain             2
Sumatra Lintong           1
Ethiopian Heirlooms       1
Moka Peaberry             1
Sulawesi                  1
Marigojipe                1
Pache Comun               1
Name: count, dtype: int64

พบคู่ที่ควรรวม: `V. Colombia`→`Colombia`, `Tico`→`Typica` (ชื่อเรียกท้องถิ่นของ Typica),
`Bourbon Ponan`→`Bourbon` — ส่วนกลุ่มอื่นที่ชื่อคล้ายแต่**ไม่รวม** (เช่น `Caturra` กับ `Catimor`
เป็นพันธุ์ต่างกันจริง)

In [8]:
fix_variety = {"V. Colombia": "Colombia", "Tico": "Typica", "Bourbon Ponan": "Bourbon"}
coffee["Variety"] = coffee["Variety"].replace(fix_variety)
print("เหลือ", coffee["Variety"].nunique(), "กลุ่ม")

เหลือ 29 กลุ่ม


**2.1** รวมเฉพาะที่มีหลักฐานว่าเป็นพันธุ์เดียวกัน (ชื่อเรียกท้องถิ่น/ตัวย่อ) — อันที่ไม่แน่ใจ
เช่นชื่อคล้ายแต่อาจต่างพันธุ์จริง ปล่อยไว้เป็นกลุ่มของตัวเองดีกว่ารวมผิด

**2.2** บอกจำนวนแถวที่แปลงได้/ไม่ได้ และเหตุผล (หน่วยปน, ช่วงค่า) — ถ้าจำนวนที่หายไม่ถึง ~10%
ให้ตัดทิ้งพร้อมหมายเหตุในรายงาน

## ส่วนที่ 3 — ไฟล์ไทย: Encoding + วันที่แปลก (PCD 2554)

In [9]:
legacy = pd.read_csv(
    repo_root() / "datasets" / "pcd_pm25_legacy" / "aqpm2.5bangkokmetropolitanregion2556.csv",
    encoding="tis-620",   # มาตรฐานรหัสอักขระภาษาไทย
)
legacy.head(3)

,Date,54T,27T,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,1/1/2013,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1/2/2013,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1/3/2013,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
data = legacy[legacy.iloc[:, 0].astype("string").str.match(r"\d", na=False)].copy()
long = data.melt(id_vars=data.columns[0], var_name="date_raw", value_name="pm25")
long["date"] = pd.to_datetime(long["date_raw"], format="%m/%d/%Y", errors="coerce")
print("แถวข้อมูล:", len(data), "| วันที่ parse ไม่ได้:", long["date"].isna().sum())
long.dropna(subset=["date"]).head(3)

แถวข้อมูล: 365 | วันที่ parse ไม่ได้: 5475


,Date,date_raw,pm25,date


**3.1** `errors="coerce"` เปลี่ยนวันที่ที่ parse ไม่ได้เป็น `NaT` แล้ว**นับได้** (`isna().sum()`)
— แถวพวงนี้กลายเป็น "รายการที่ต้องไปดูต่อ" แทนที่จะทำโปรแกรมพังกลางทาง
การปล่อยเข้มงวด (raise) ดีตอนสร้าง pipeline ที่ต้องการความแน่ใจสูง แต่สำหรับข้อมูลสกปรกแบบนี้
"หยุดแล้วรายงาน" (`coerce` + นับ) ส่งต่อให้เพื่อนร่วมทีมได้ปลอดภัยกว่า

## ส่วนที่ 4 — เช็คลิสต์ของคุณเอง

ไม่มีคำตอบถูก/ผิด — จุดประสงค์คือให้ผู้เรียนเชื่อม 4 ปัญหาที่เพิ่งแก้ไปกับไฟล์จริงในงานตัวเอง